# S33. Estimação da Média Populacional
## Estimation of the Population Mean

[◀ Anterior](S32_Estimacao_Proporcao.ipynb) | [📖 Índice](S00_Index_Estatistica.ipynb) | [Próximo ▶](S34_Testes_de_Hipotese.ipynb)

## 🎯 Objetivos de Aprendizagem

- Compreender a estimação da média populacional (μ)
- Distinguir quando usar Z ou t para o intervalo de confiança
- Calcular o IC para a média com Python
- Interpretar corretamente o IC da média
- Aplicar a dados reais (altura, peso, etc.)

## 📝 Introdução

A **média populacional (μ)** é um dos parâmetros mais importantes em estatística. Queremos saber: qual a altura média dos brasileiros? Qual o peso médio dos recém-nascidos? Qual o salário médio dos programadores?

Como não podemos medir toda a população, usamos a média de uma amostra (x̄) para estimar μ. O **intervalo de confiança para a média** nos dá uma faixa de valores plausíveis, levando em conta a variabilidade amostral.

Este capítulo consolida os conceitos dos capítulos anteriores (distribuição normal, t de Student, estimação) aplicados especificamente à estimação da média.

## 📚 Explicação Detalhada

### 1. Intervalo de Confiança para a Média

**Quando σ é conhecido (ou n ≥ 30):**

$$IC = \bar{x} \pm z_{\alpha/2} \cdot \frac{\sigma}{\sqrt{n}}$$

**Quando σ é desconhecido (padrão):**

$$IC = \bar{x} \pm t_{\alpha/2, n-1} \cdot \frac{s}{\sqrt{n}}$$

### 2. Escolha entre Z e t

| Situação | Distribuição | Motivo |
|----------|-------------|--------|
| σ conhecido (populacional) | Z (normal) | A estatística padronizada segue N(0,1) exatamente |
| σ desconhecido, n grande | t (≈Z) | s ≈ σ, então t ≈ Z, mas t é mais correto |
| σ desconhecido, n pequeno | t (Student) | A incerteza adicional exige caudas mais pesadas |

### 3. Interpretação

Um IC de 95% para a média significa que, se repetíssemos o processo de amostragem muitas vezes, 95% dos intervalos construídos conteriam a verdadeira média populacional.

In [ ]:
# Importando bibliotecas
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

print("Bibliotecas importadas com sucesso!")

## 💻 Exemplos Práticos

In [ ]:
# Exemplo 1: Estimando a altura média de mulheres brasileiras
# Dados reais aproximados: altura de mulheres ~ N(1.62, 0.06)

np.random.seed(42)

# Coletamos uma amostra de 50 mulheres
alturas = np.random.normal(loc=1.62, scale=0.06, size=50)

n = len(alturas)
media_alt = np.mean(alturas)
desvio_alt = np.std(alturas, ddof=1)
ep_alt = desvio_alt / np.sqrt(n)

print("=== Estimação da Altura Média ===")
print(f"Tamanho da amostra: {n}")
print(f"Média amostral (x̄): {media_alt:.4f} m")
print(f"Desvio padrão amostral (s): {desvio_alt:.4f} m")
print(f"Erro padrão (s/√n): {ep_alt:.4f} m")

# IC 95% usando t
ic_95_alt = stats.t.interval(0.95, df=n-1, loc=media_alt, scale=ep_alt)
print(f"\nIC 95% para altura média: [{ic_95_alt[0]:.4f}, {ic_95_alt[1]:.4f}] m")
print(f"IC 95%: [{ic_95_alt[0]:.3f}, {ic_95_alt[1]:.3f}] m")

# Comparação com Z
z_crit = stats.norm.ppf(0.975)
ic_z_alt = (media_alt - z_crit*ep_alt, media_alt + z_crit*ep_alt)
print(f"\nSe usássemos Z (n=50, diferença pequena):")
print(f"IC 95% (Z): [{ic_z_alt[0]:.4f}, {ic_z_alt[1]:.4f}] m")

In [ ]:
# Exemplo 2: Estimando o peso médio de recém-nascidos
# População real: peso ~ N(3.2 kg, 0.5 kg)

np.random.seed(123)

pesos = np.random.normal(loc=3.2, scale=0.5, size=30)

n_peso = len(pesos)
media_peso = np.mean(pesos)
s_peso = np.std(pesos, ddof=1)
ep_peso = s_peso / np.sqrt(n_peso)

ic_95_peso = stats.t.interval(0.95, n_peso-1, media_peso, ep_peso)
ic_99_peso = stats.t.interval(0.99, n_peso-1, media_peso, ep_peso)
ic_90_peso = stats.t.interval(0.90, n_peso-1, media_peso, ep_peso)

print("=== Estimação do Peso de Recém-Nascidos ===")
print(f"Amostra: n = {n_peso}")
print(f"Média: {media_peso:.3f} kg")
print(f"Desvio padrão amostral: {s_peso:.3f} kg")
print()
print(f"IC 90%: [{ic_90_peso[0]:.3f}, {ic_90_peso[1]:.3f}] kg")
print(f"IC 95%: [{ic_95_peso[0]:.3f}, {ic_95_peso[1]:.3f}] kg")
print(f"IC 99%: [{ic_99_peso[0]:.3f}, {ic_99_peso[1]:.3f}] kg")

In [ ]:
# Exemplo 3: Visualização do IC para diferentes níveis de confiança

niveis = [0.80, 0.85, 0.90, 0.95, 0.99]
ics = [stats.t.interval(nv, n_peso-1, media_peso, ep_peso) for nv in niveis]

plt.figure(figsize=(10, 5))

for i, (nv, ic) in enumerate(zip(niveis, ics)):
    largura = ic[1] - ic[0]
    plt.plot([ic[0], ic[1]], [i, i], 'o-', lw=2, markersize=6)
    plt.text(ic[1] + 0.01, i, f'IC {nv:.0%} (largura={largura:.3f})', 
             va='center', fontsize=10)

plt.axvline(media_peso, color='black', linestyle='--', 
            label=f'Média amostral = {media_peso:.3f}', lw=1.5)
plt.yticks(range(len(niveis)), [f'{nv:.0%}' for nv in niveis])
plt.xlabel('Peso (kg)', fontsize=12)
plt.title('Intervalos de Confiança para o Peso Médio', fontsize=13)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Exemplo 4: Aplicação com n pequeno (n = 10)
# Salários de estagiários (R$)

salarios = np.array([1500, 1800, 1200, 2000, 1600, 
                     1400, 1750, 1550, 1900, 1650])

n_sal = len(salarios)
media_sal = np.mean(salarios)
s_sal = np.std(salarios, ddof=1)
ep_sal = s_sal / np.sqrt(n_sal)

ic_95_sal = stats.t.interval(0.95, n_sal-1, media_sal, ep_sal)

print("=== Estimação de Salário Médio (n pequeno) ===")
print(f"Dados: {salarios}")
print(f"n = {n_sal}")
print(f"Média amostral: R$ {media_sal:.2f}")
print(f"Desvio padrão amostral: R$ {s_sal:.2f}")
print(f"Erro padrão: R$ {ep_sal:.2f}")
print(f"Graus de liberdade: {n_sal - 1}")
print(f"Valor crítico t (95%): {stats.t.ppf(0.975, n_sal-1):.4f}")
print(f"IC 95% para salário médio: [R$ {ic_95_sal[0]:.2f}, R$ {ic_95_sal[1]:.2f}]")

In [ ]:
# Exemplo 5: Verificação via simulação — cobertura do IC para média
# Verificamos se realmente ~95% dos ICs contêm μ

np.random.seed(42)

media_verdadeira = 100
desvio_verdadeiro = 20
n_sim = 30
n_repeticoes = 5000

contadores = 0

for _ in range(n_repeticoes):
    amostra_sim = np.random.normal(media_verdadeira, desvio_verdadeiro, n_sim)
    x_bar = np.mean(amostra_sim)
    s_sim = np.std(amostra_sim, ddof=1)
    
    ic_sim = stats.t.interval(0.95, n_sim-1, x_bar, s_sim / np.sqrt(n_sim))
    
    if ic_sim[0] <= media_verdadeira <= ic_sim[1]:
        contadores += 1

proporcao_cobertura = contadores / n_repeticoes

print("=== Simulação de Cobertura do IC ===")
print(f"Média verdadeira: {media_verdadeira}")
print(f"n = {n_sim}")
print(f"Repetições: {n_repeticoes}")
print(f"ICs que contêm μ: {contadores} de {n_repeticoes}")
print(f"Proporção de cobertura: {proporcao_cobertura:.4f} ({proporcao_cobertura:.1%})")
print(f"Esperado (nominal): 95%")

In [ ]:
# Exemplo 6: Comparação entre usar t e Z com n pequeno
# Dados: amostra de 12 observações

np.random.seed(123)
dados_peq = np.random.normal(50, 10, 12)

n_peq = len(dados_peq)
media_peq = np.mean(dados_peq)
s_peq = np.std(dados_peq, ddof=1)
ep_peq = s_peq / np.sqrt(n_peq)

# IC com t
ic_t = stats.t.interval(0.95, n_peq-1, media_peq, ep_peq)

# IC com Z (incorreto para n pequeno)
z_val = stats.norm.ppf(0.975)
ic_z = (media_peq - z_val*ep_peq, media_peq + z_val*ep_peq)

largura_t = ic_t[1] - ic_t[0]
largura_z = ic_z[1] - ic_z[0]

print("=== Comparação t vs Z (n = 12) ===")
print(f"Média: {media_peq:.2f}")
print(f"s = {s_peq:.2f}, EP = {ep_peq:.2f}")
print(f"\nIC 95% (t):       [{ic_t[0]:.2f}, {ic_t[1]:.2f}] → largura = {largura_t:.2f}")
print(f"IC 95% (Z):       [{ic_z[0]:.2f}, {ic_z[1]:.2f}] → largura = {largura_z:.2f}")
print(f"\nDiferença: o IC com t é {largura_t / largura_z - 1:.1%} mais largo (mais conservador).")
print("Com n pequeno, usar t é ESSENCIAL.")

In [ ]:
# Exemplo 7: Interpretação gráfica do IC
# Histograma com IC sobreposto

plt.figure(figsize=(10, 5))

plt.hist(salarios, bins=8, color='skyblue', edgecolor='black', alpha=0.7, density=True)

# Curva t ajustada
x_plot = np.linspace(min(salarios) - 200, max(salarios) + 200, 200)
y_plot = stats.t.pdf((x_plot - media_sal) / ep_sal, df=n_sal-1) / ep_sal
plt.plot(x_plot, y_plot, 'r-', lw=2, label='Distribuição t ajustada')

# IC
plt.axvline(media_sal, color='blue', linestyle='-', lw=2, label=f'Média = R$ {media_sal:.0f}')
plt.axvline(ic_95_sal[0], color='green', linestyle='--', lw=1.5, 
            label=f'IC 95%: R$ {ic_95_sal[0]:.0f} a R$ {ic_95_sal[1]:.0f}')
plt.axvline(ic_95_sal[1], color='green', linestyle='--', lw=1.5)

plt.title('Distribuição dos Salários com IC 95% para a Média', fontsize=13)
plt.xlabel('Salário (R$)', fontsize=12)
plt.ylabel('Densidade', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## ✅ Exercícios Resolvidos

**Exercício 1:** Uma amostra de 36 alunos tem IMC médio de 24.5 com desvio padrão de 3.2. Construa o IC de 95% para o IMC médio populacional.

In [ ]:
# Solução Exercício 1
n = 36
media_imc = 24.5
s_imc = 3.2
ep_imc = s_imc / np.sqrt(n)

ic_imc = stats.t.interval(0.95, n-1, media_imc, ep_imc)

print(f"n = {n}, x̄ = {media_imc}, s = {s_imc}")
print(f"Erro padrão: {ep_imc:.3f}")
print(f"IC 95% para IMC médio: [{ic_imc[0]:.2f}, {ic_imc[1]:.2f}]")
print(f"\nInterpretação: Estamos 95% confiantes de que o IMC médio
populacional está entre {ic_imc[0]:.2f} e {ic_imc[1]:.2f}.")

**Exercício 2:** Com n = 10, uma amostra tem média 50 e s = 8. Qual a diferença entre usar t e Z para o IC de 95%?

In [ ]:
# Solução Exercício 2
n = 10
media = 50
s = 8
ep = s / np.sqrt(n)

ic_t_ex = stats.t.interval(0.95, n-1, media, ep)
ic_z_ex = (media - stats.norm.ppf(0.975)*ep, media + stats.norm.ppf(0.975)*ep)

larg_t = ic_t_ex[1] - ic_t_ex[0]
larg_z = ic_z_ex[1] - ic_z_ex[0]

print(f"IC 95% (t): [{ic_t_ex[0]:.2f}, {ic_t_ex[1]:.2f}] (largura = {larg_t:.2f})")
print(f"IC 95% (Z): [{ic_z_ex[0]:.2f}, {ic_z_ex[1]:.2f}] (largura = {larg_z:.2f})")
print(f"\nDiferença: {(larg_t/larg_z - 1)*100:.1f}% mais largo com t")
print(f"Neste caso, usar Z subestimaria a incerteza em {(1 - larg_z/larg_t)*100:.1f}%.")

## 📋 Exercícios Propostos

| Nível | Exercício |
|-------|-----------|
| 🟢 Fácil | 1. Uma amostra de 25 pessoas tem QI médio 105 e s = 12. Calcule o IC 95% para o QI médio populacional. |
| 🟡 Médio | 2. Os tempos (em min) de 8 corredores nos 5km foram: [18.2, 19.5, 17.8, 20.1, 18.9, 19.0, 17.5, 20.5]. Calcule o IC 95% para o tempo médio. |
| 🔴 Difícil | 3. Simule 1000 amostras de n=15 de uma população exponencial (não normal). Verifique quantos ICs (usando t) contêm a verdadeira média. A cobertura ainda é ~95%? O que isso nos diz sobre a robustez do IC? |

In [ ]:
# 🟢 Exercício 1: Escreva sua solução aqui
import numpy as np
from scipy import stats

# Sua solução


In [ ]:
# 🟡 Exercício 2: Escreva sua solução aqui
tempos = np.array([18.2, 19.5, 17.8, 20.1, 18.9, 19.0, 17.5, 20.5])

# Sua solução


## 🏆 Desafios

1. Explique o conceito de robustez do IC baseado em t quando a população não é normal.
2. Implemente uma função que calcule o tamanho de amostra necessário para estimar a média com uma margem de erro específica.
3. Pesquise sobre o IC bootstrap para a média e implemente uma versão simples.

## 📌 Resumo

- **IC para média (σ desconhecido)**: $\bar{x} \pm t \cdot s/\sqrt{n}$.
- Use **t de Student** quando σ é desconhecido (quase sempre).
- Com n ≥ 30, a diferença entre t e Z é pequena.
- O IC nos dá uma faixa de valores plausíveis para μ.
- `stats.t.interval(confidence, df, loc, scale)` calcula diretamente.

## 💡 Curiosidades

Você sabia que a altura média dos brasileiros aumentou nas últimas décadas? Segundo dados do IBGE, homens nascidos na década de 1990 são em média 2.5 cm mais altos que os nascidos na década de 1960. Isso reflete melhorias na nutrição e saúde pública. Estudos como esse dependem de intervalos de confiança para afirmar que a diferença é real e não apenas devido ao acaso amostral.

## ✅ Boas Práticas

1. Sempre use `ddof=1` ao calcular o desvio padrão amostral.
2. Use `stats.t.interval()` para calcular o IC diretamente.
3. Prefira t sobre Z quando σ for desconhecido (mesmo com n grande).
4. Para n < 15, verifique a suposição de normalidade dos dados.
5. Reporte o IC junto com a média: x̄ = 24.5 (IC 95%: 23.4 - 25.6).

## ⚠️ Erros Comuns

| Erro | Como Evitar |
|------|-------------|
| Usar Z mesmo sem conhecer σ | Com σ desconhecido, SEMPRE use t. |
| Interpretar IC como "95% dos dados" | IC é para o parâmetro, não para os dados individuais. |
| Ignorar a suposição de normalidade | Para n pequeno, verifique a distribuição dos dados. |
| Esquecer que IC depende do tamanho da amostra | n maior → IC mais estreito (mais preciso). |

## 📖 Referências

- [W3Statistics — Mean Confidence Interval](https://www.w3schools.com/statistics/statistics_mean_confidence_interval.php)
- [Khan Academy — Confidence Intervals (Means)](https://pt.khanacademy.org/math/statistics-probability/confidence-intervals-one-sample)
- [SciPy — t.interval](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.t.html)
- [IBGE — Estatísticas de Gênero](https://www.ibge.gov.br/apps/snig/v1/index.html)

[◀ Anterior](S32_Estimacao_Proporcao.ipynb) | [📖 Índice](S00_Index_Estatistica.ipynb) | [Próximo ▶](S34_Testes_de_Hipotese.ipynb)